In [0]:
CREATE OR REPLACE TABLE production.supply_analytics.option_price_discrepancy AS

WITH pricing_base AS (
  SELECT 
    supplier_id,
    segment,
    tour_id,
    tour_option_id,
    pricing_id,
    tour_title,
    tour_option_title,
    location_name,
    activity_type,
    tour_category,
    from_price_without_deal,
    currency_iso_code,
    individual_categories,
    group_categories,
    has_individual_pricing,
    has_group_pricing
  FROM production.supply_analytics.pricing_feature_audit_base
  WHERE uses_external_pricing = 0
    AND from_price_without_deal IS NOT NULL
    AND from_price_without_deal > 0
),

-- GET LATEST EXCHANGE RATES
latest_exchange_rates AS (
  SELECT 
    c.iso_code,
    er.exchange_rate
  FROM production.dwh.dim_currency c
  INNER JOIN production.dwh.dim_currency_all_dates er
    ON c.currency_id = er.currency_id
  WHERE er.update_timestamp = CURRENT_DATE()
),

pricing_with_fx AS (
  SELECT 
    pb.*,
    COALESCE(ler.exchange_rate, 1.0) as exchange_rate,
    pb.from_price_without_deal / COALESCE(ler.exchange_rate, 1.0) as from_price_eur
  FROM pricing_base pb
  LEFT JOIN latest_exchange_rates ler
    ON pb.currency_iso_code = ler.iso_code
),

-- GET BASE RETAIL PRICE - ADULT CATEGORY ONLY FOR INDIVIDUAL PRICING
base_retail_price_individual AS (
  SELECT 
    tour_option_id,
    pricing_id,
    MIN(price_tier.retailPrice) as base_retail_price,
    MIN(price_tier.retailPrice / exchange_rate) as base_retail_price_eur
  FROM pricing_with_fx pb
  LATERAL VIEW OUTER EXPLODE(pb.individual_categories) AS ind_cat
  LATERAL VIEW OUTER EXPLODE(ind_cat.prices) AS price_tier
  WHERE pb.has_individual_pricing = 1
    AND UPPER(ind_cat.ticketCategory) = 'ADULT'
  GROUP BY tour_option_id, pricing_id
),

-- GET BASE RETAIL PRICE - SMALLEST GROUP PRICE
base_retail_price_group AS (
  SELECT 
    tour_option_id,
    pricing_id,
    MIN(price_tier.retailPrice) as base_retail_price,
    MIN(price_tier.retailPrice / exchange_rate) as base_retail_price_eur
  FROM pricing_with_fx pb
  LATERAL VIEW OUTER EXPLODE(pb.group_categories) AS grp_cat
  LATERAL VIEW OUTER EXPLODE(grp_cat.prices) AS price_tier
  WHERE pb.has_group_pricing = 1
    AND grp_cat.isAdditionalPaxForGroup = false
  GROUP BY tour_option_id, pricing_id
),

-- COMBINE RETAIL PRICES
option_prices AS (
  SELECT 
    pw.supplier_id,
    pw.segment,
    pw.tour_id,
    pw.tour_option_id,
    pw.pricing_id,
    pw.tour_title,
    pw.tour_option_title,
    pw.location_name,
    pw.activity_type,
    pw.tour_category,
    pw.currency_iso_code,
    pw.exchange_rate,
    pw.from_price_without_deal,
    pw.from_price_eur,
    pw.has_individual_pricing,
    pw.has_group_pricing,
    
    COALESCE(brpi.base_retail_price, brpg.base_retail_price) as base_retail_price,
    COALESCE(brpi.base_retail_price_eur, brpg.base_retail_price_eur) as base_retail_price_eur,
    
    CASE 
      WHEN brpi.base_retail_price IS NOT NULL THEN 'ADULT_INDIVIDUAL'
      WHEN brpg.base_retail_price IS NOT NULL THEN 'SMALLEST_GROUP'
      ELSE NULL
    END as pricing_type_used
    
  FROM pricing_with_fx pw
  LEFT JOIN base_retail_price_individual brpi 
    ON pw.tour_option_id = brpi.tour_option_id AND pw.pricing_id = brpi.pricing_id
  LEFT JOIN base_retail_price_group brpg 
    ON pw.tour_option_id = brpg.tour_option_id AND pw.pricing_id = brpg.pricing_id
),

-- CALCULATE GLOBAL BENCHMARKS
global_benchmarks AS (
  SELECT 
    PERCENTILE(from_price_eur, 0.50) as global_median_eur,
    PERCENTILE(from_price_eur, 0.75) as global_p75_eur,
    PERCENTILE(from_price_eur, 0.90) as global_p90_eur,
    PERCENTILE(from_price_eur, 0.95) as global_p95_eur,
    PERCENTILE(from_price_eur, 0.99) as global_p99_eur,
    AVG(from_price_eur) as global_avg_eur,
    STDDEV(from_price_eur) as global_stddev_eur
  FROM option_prices
),

-- CALCULATE LOCATION BENCHMARKS
location_benchmarks AS (
  SELECT 
    location_name,
    COUNT(DISTINCT tour_id) as num_tours,
    PERCENTILE(from_price_eur, 0.50) as location_median_eur,
    PERCENTILE(from_price_eur, 0.75) as location_p75_eur,
    PERCENTILE(from_price_eur, 0.90) as location_p90_eur,
    PERCENTILE(from_price_eur, 0.95) as location_p95_eur,
    AVG(from_price_eur) as location_avg_eur,
    STDDEV(from_price_eur) as location_stddev_eur
  FROM option_prices
  GROUP BY location_name
  HAVING COUNT(DISTINCT tour_id) >= 10
),

-- CALCULATE ACTIVITY TYPE BENCHMARKS
activity_benchmarks AS (
  SELECT 
    activity_type,
    COUNT(DISTINCT tour_id) as num_tours,
    PERCENTILE(from_price_eur, 0.50) as activity_median_eur,
    PERCENTILE(from_price_eur, 0.75) as activity_p75_eur,
    PERCENTILE(from_price_eur, 0.90) as activity_p90_eur,
    PERCENTILE(from_price_eur, 0.95) as activity_p95_eur,
    AVG(from_price_eur) as activity_avg_eur,
    STDDEV(from_price_eur) as activity_stddev_eur
  FROM option_prices
  GROUP BY activity_type
  HAVING COUNT(DISTINCT tour_id) >= 10
),

-- CALCULATE LOCATION + ACTIVITY TYPE BENCHMARKS
location_activity_benchmarks AS (
  SELECT 
    location_name,
    activity_type,
    COUNT(DISTINCT tour_id) as num_tours,
    PERCENTILE(from_price_eur, 0.50) as loc_act_median_eur,
    PERCENTILE(from_price_eur, 0.90) as loc_act_p90_eur,
    PERCENTILE(from_price_eur, 0.95) as loc_act_p95_eur,
    AVG(from_price_eur) as loc_act_avg_eur
  FROM option_prices
  GROUP BY location_name, activity_type
  HAVING COUNT(DISTINCT tour_id) >= 5
),

-- CALCULATE LOCATION + ACTIVITY TYPE + CATEGORY BENCHMARKS
location_activity_category_benchmarks AS (
  SELECT 
    location_name,
    activity_type,
    tour_category,
    COUNT(DISTINCT tour_id) as num_tours,
    PERCENTILE(from_price_eur, 0.50) as loc_act_cat_median_eur,
    PERCENTILE(from_price_eur, 0.90) as loc_act_cat_p90_eur,
    PERCENTILE(from_price_eur, 0.95) as loc_act_cat_p95_eur,
    AVG(from_price_eur) as loc_act_cat_avg_eur
  FROM option_prices
  GROUP BY location_name, activity_type, tour_category
  HAVING COUNT(DISTINCT tour_id) >= 3
),

-- TOUR-LEVEL AGGREGATES
tour_aggregates AS (
  SELECT 
    tour_id,
    COUNT(DISTINCT tour_option_id) as num_options,
    COUNT(DISTINCT from_price_eur) as num_distinct_prices,
    MIN(from_price_eur) as tour_min_price_eur,
    MAX(from_price_eur) as tour_max_price_eur,
    AVG(from_price_eur) as tour_avg_price_eur,
    STDDEV(from_price_eur) as tour_stddev_price_eur,
    CASE 
      WHEN AVG(from_price_eur) > 0 
      THEN STDDEV(from_price_eur) / AVG(from_price_eur)
      ELSE 0
    END as tour_price_cv,
    CASE 
      WHEN MIN(from_price_eur) > 0 
      THEN (MAX(from_price_eur) - MIN(from_price_eur)) / MIN(from_price_eur)
      ELSE 0
    END as tour_price_spread_ratio
  FROM option_prices
  GROUP BY tour_id
),

-- COMBINE ALL DATA
options_with_benchmarks AS (
  SELECT 
    op.*,
    gb.global_median_eur,
    gb.global_p90_eur,
    gb.global_p95_eur,
    gb.global_p99_eur,
    gb.global_avg_eur,
    gb.global_stddev_eur,
    lb.location_median_eur,
    lb.location_p90_eur,
    lb.location_p95_eur,
    lb.location_avg_eur,
    ab.activity_median_eur,
    ab.activity_p90_eur,
    ab.activity_p95_eur,
    ab.activity_avg_eur,
    lab.loc_act_median_eur,
    lab.loc_act_p90_eur,
    lab.loc_act_p95_eur,
    lacb.loc_act_cat_median_eur,
    lacb.loc_act_cat_p90_eur,
    lacb.loc_act_cat_p95_eur,
    ta.num_options,
    ta.num_distinct_prices,
    ta.tour_min_price_eur,
    ta.tour_max_price_eur,
    ta.tour_avg_price_eur,
    ta.tour_price_cv,
    ta.tour_price_spread_ratio
  FROM option_prices op
  CROSS JOIN global_benchmarks gb
  LEFT JOIN location_benchmarks lb ON op.location_name = lb.location_name
  LEFT JOIN activity_benchmarks ab ON op.activity_type = ab.activity_type
  LEFT JOIN location_activity_benchmarks lab 
    ON op.location_name = lab.location_name AND op.activity_type = lab.activity_type
  LEFT JOIN location_activity_category_benchmarks lacb 
    ON op.location_name = lacb.location_name 
    AND op.activity_type = lacb.activity_type 
    AND op.tour_category = lacb.tour_category
  LEFT JOIN tour_aggregates ta ON op.tour_id = ta.tour_id
),

-- ADD WITHIN-TOUR PERCENTILES TO OPTIONS
options_with_tour_percentiles AS (
  SELECT 
    owb.*,
    PERCENT_RANK() OVER (PARTITION BY owb.tour_id ORDER BY owb.from_price_eur) as price_percentile_within_tour
  FROM options_with_benchmarks owb
),

-- BUCKET 1: ABNORMALLY HIGH FROM PRICES
bucket1_high_from_prices AS (
  SELECT 
    supplier_id,
    segment,
    tour_id,
    tour_option_id,
    pricing_id,
    tour_title,
    tour_option_title,
    location_name,
    activity_type,
    tour_category,
    currency_iso_code,
    from_price_without_deal,
    from_price_eur,
    exchange_rate,
    has_individual_pricing,
    has_group_pricing,
    
    'BUCKET_1_HIGH_FROM_PRICE' as issue_bucket,
    NULL as sub_bucket,
    
    CASE 
      WHEN loc_act_cat_p95_eur IS NOT NULL AND from_price_eur > loc_act_cat_p95_eur 
      THEN CONCAT('From price ', ROUND(from_price_eur, 2), ' EUR is ', 
                  ROUND(100.0 * (from_price_eur / loc_act_cat_p95_eur - 1), 0), 
                  '% above P95 for ', location_name, ' - ', activity_type, ' - ', tour_category, 
                  ' (P95: ', ROUND(loc_act_cat_p95_eur, 2), ' EUR)')
      
      WHEN loc_act_p95_eur IS NOT NULL AND from_price_eur > loc_act_p95_eur 
      THEN CONCAT('From price ', ROUND(from_price_eur, 2), ' EUR is ', 
                  ROUND(100.0 * (from_price_eur / loc_act_p95_eur - 1), 0), 
                  '% above P95 for ', location_name, ' - ', activity_type, 
                  ' (P95: ', ROUND(loc_act_p95_eur, 2), ' EUR). Category: ', tour_category)
      
      WHEN location_p95_eur IS NOT NULL AND from_price_eur > location_p95_eur 
      THEN CONCAT('From price ', ROUND(from_price_eur, 2), ' EUR is ', 
                  ROUND(100.0 * (from_price_eur / location_p95_eur - 1), 0), 
                  '% above P95 for ', location_name, 
                  ' (P95: ', ROUND(location_p95_eur, 2), ' EUR). Activity: ', activity_type, ', Category: ', tour_category)
      ELSE NULL
    END as problem_description,
    
    CASE 
      WHEN loc_act_cat_p95_eur IS NOT NULL AND from_price_eur > loc_act_cat_p95_eur THEN 10
      WHEN loc_act_p95_eur IS NOT NULL AND from_price_eur > loc_act_p95_eur THEN 9
      WHEN location_p95_eur IS NOT NULL AND from_price_eur > location_p95_eur THEN 8
      ELSE 0
    END as severity_score,
    
    COALESCE(loc_act_cat_median_eur, loc_act_median_eur, location_median_eur) as benchmark_median_eur,
    COALESCE(loc_act_cat_p95_eur, loc_act_p95_eur, location_p95_eur) as benchmark_p95_eur,
    
    NULL as pricing_type_used,
    NULL as tour_min_price_eur,
    NULL as tour_max_price_eur,
    NULL as tour_price_spread_pct,
    NULL as base_retail_price,
    NULL as base_retail_price_eur,
    NULL as price_gap_pct
    
  FROM options_with_tour_percentiles
  WHERE 
    (loc_act_cat_p95_eur IS NOT NULL AND from_price_eur > loc_act_cat_p95_eur)
    OR (loc_act_p95_eur IS NOT NULL AND from_price_eur > loc_act_p95_eur)
    OR (location_p95_eur IS NOT NULL AND from_price_eur > location_p95_eur)
),

-- BUCKET 2: WITHIN-TOUR PRICING ISSUES (UNIFIED WITH SUB-TYPES)
bucket2_tour_issues AS (
  SELECT 
    owtp.supplier_id,
    owtp.segment,
    owtp.tour_id,
    owtp.tour_option_id,
    owtp.pricing_id,
    owtp.tour_title,
    owtp.tour_option_title,
    owtp.location_name,
    owtp.activity_type,
    owtp.tour_category,
    owtp.currency_iso_code,
    owtp.from_price_without_deal,
    owtp.from_price_eur,
    owtp.exchange_rate,
    owtp.has_individual_pricing,
    owtp.has_group_pricing,
    
    'BUCKET_2_TOUR_VARIANCE' as issue_bucket,
    
    -- SUB-BUCKET CLASSIFICATION
    CASE 
      -- Sub-type A: Duplicate pricing configs (same title, multiple prices)
      WHEN title_stats.num_pricing_configs > 1 AND title_stats.num_distinct_prices > 1
      THEN 'DUPLICATE_PRICING_CONFIGS'
      
      -- Sub-type B: High price with no demand (market rejection)
      WHEN COALESCE(perf.bookings_l12mo, 0) = 0 AND owtp.from_price_eur > owtp.tour_avg_price_eur * 1.2
      THEN 'HIGH_PRICE_NO_DEMAND'
      
      WHEN COALESCE(perf.bookings_l12mo, 0) > 0 AND COALESCE(perf.bookings_l12mo, 0) <= 5 
           AND owtp.from_price_eur > owtp.tour_avg_price_eur * 1.5
      THEN 'HIGH_PRICE_LOW_DEMAND'
      
      -- Sub-type C: Similar titles, different prices (unclear differentiation)
      WHEN similar_title.has_similar_title = 1 AND similar_title.price_diff_pct > 0.3
      THEN 'UNCLEAR_DIFFERENTIATION'
      
      -- Sub-type D: High percentile within tour (original logic)
      WHEN owtp.price_percentile_within_tour >= 0.75 AND owtp.tour_price_spread_ratio > 0.5
      THEN 'HIGH_VARIANCE_OUTLIER'
      
      ELSE 'OTHER'
    END as sub_bucket,
    
    -- DYNAMIC PROBLEM DESCRIPTION BASED ON SUB-TYPE
    CASE 
      WHEN title_stats.num_pricing_configs > 1 AND title_stats.num_distinct_prices > 1
      THEN CONCAT('Option title "', owtp.tour_option_title, '" has ', title_stats.num_pricing_configs, 
                  ' pricing configs with prices ', ROUND(title_stats.min_price_eur, 2), '-', 
                  ROUND(title_stats.max_price_eur, 2), ' EUR (',
                  ROUND(100.0 * (title_stats.max_price_eur - title_stats.min_price_eur) / title_stats.min_price_eur, 0),
                  '% spread). This config: ', ROUND(owtp.from_price_eur, 2), ' EUR')
      
      WHEN COALESCE(perf.bookings_l12mo, 0) = 0 AND owtp.from_price_eur > owtp.tour_avg_price_eur * 1.2
      THEN CONCAT('Option at ', ROUND(owtp.from_price_eur, 2), ' EUR (',
                  ROUND(100.0 * (owtp.from_price_eur / owtp.tour_avg_price_eur - 1), 0),
                  '% above tour avg) has 0 bookings L12M. Tour range: ', 
                  ROUND(owtp.tour_min_price_eur, 2), '-', ROUND(owtp.tour_max_price_eur, 2), ' EUR')
      
      WHEN COALESCE(perf.bookings_l12mo, 0) > 0 AND COALESCE(perf.bookings_l12mo, 0) <= 5 
           AND owtp.from_price_eur > owtp.tour_avg_price_eur * 1.5
      THEN CONCAT('Option at ', ROUND(owtp.from_price_eur, 2), ' EUR (',
                  ROUND(100.0 * (owtp.from_price_eur / owtp.tour_avg_price_eur - 1), 0),
                  '% above tour avg) has only ', perf.bookings_l12mo, ' bookings L12M. Tour range: ', 
                  ROUND(owtp.tour_min_price_eur, 2), '-', ROUND(owtp.tour_max_price_eur, 2), ' EUR')
      
      WHEN similar_title.has_similar_title = 1 AND similar_title.price_diff_pct > 0.3
      THEN CONCAT('Option "', owtp.tour_option_title, '" (', ROUND(owtp.from_price_eur, 2),
                  ' EUR) has similar title to "', similar_title.other_title, '" (',
                  ROUND(similar_title.other_price_eur, 2), ' EUR) but ', 
                  ROUND(similar_title.price_diff_pct * 100, 0), '% price difference')
      
      WHEN owtp.price_percentile_within_tour >= 0.75 AND owtp.tour_price_spread_ratio > 0.5
      THEN CONCAT('Tour has ', owtp.num_options, ' options with ', 
                  ROUND(owtp.tour_price_spread_ratio * 100, 0), '% price spread. This option (', 
                  ROUND(owtp.from_price_eur, 2), ' EUR) is at P', ROUND(owtp.price_percentile_within_tour * 100, 0),
                  ' within tour. Tour range: ', ROUND(owtp.tour_min_price_eur, 2), '-', 
                  ROUND(owtp.tour_max_price_eur, 2), ' EUR')
      ELSE NULL
    END as problem_description,
    
    -- SEVERITY BASED ON SUB-TYPE
    CASE 
      WHEN title_stats.num_pricing_configs >= 4 THEN 10
      WHEN title_stats.num_pricing_configs >= 3 THEN 9
      WHEN COALESCE(perf.bookings_l12mo, 0) = 0 AND owtp.from_price_eur > owtp.tour_avg_price_eur * 1.5 THEN 10
      WHEN COALESCE(perf.bookings_l12mo, 0) = 0 AND owtp.from_price_eur > owtp.tour_avg_price_eur * 1.2 THEN 9
      WHEN COALESCE(perf.bookings_l12mo, 0) <= 5 AND owtp.from_price_eur > owtp.tour_avg_price_eur * 1.5 THEN 8
      WHEN similar_title.price_diff_pct > 1.0 THEN 9
      WHEN similar_title.price_diff_pct > 0.5 THEN 8
      WHEN owtp.price_percentile_within_tour >= 0.90 AND owtp.tour_price_spread_ratio > 1.0 THEN 10
      WHEN owtp.price_percentile_within_tour >= 0.90 AND owtp.tour_price_spread_ratio > 0.5 THEN 9
      WHEN owtp.price_percentile_within_tour >= 0.75 AND owtp.tour_price_spread_ratio > 1.0 THEN 8
      WHEN owtp.price_percentile_within_tour >= 0.75 AND owtp.tour_price_spread_ratio > 0.5 THEN 7
      ELSE 5
    END as severity_score,
    
    NULL as benchmark_median_eur,
    NULL as benchmark_p95_eur,
    NULL as pricing_type_used,
    
    owtp.tour_min_price_eur,
    owtp.tour_max_price_eur,
    ROUND(owtp.tour_price_spread_ratio * 100, 0) as tour_price_spread_pct,
    NULL as base_retail_price,
    NULL as base_retail_price_eur,
    NULL as price_gap_pct
    
  FROM options_with_tour_percentiles owtp
  
  -- JOIN: Duplicate title detection
  LEFT JOIN (
    SELECT 
      tour_id,
      tour_option_title,
      COUNT(DISTINCT pricing_id) as num_pricing_configs,
      COUNT(DISTINCT from_price_eur) as num_distinct_prices,
      MIN(from_price_eur) as min_price_eur,
      MAX(from_price_eur) as max_price_eur
    FROM options_with_tour_percentiles
    GROUP BY tour_id, tour_option_title
  ) title_stats
    ON owtp.tour_id = title_stats.tour_id 
    AND owtp.tour_option_title = title_stats.tour_option_title
  
  -- JOIN: Performance data
  LEFT JOIN production.supply_analytics.option_price_discrepancy_performance perf
    ON owtp.tour_option_id = perf.tour_option_id
    AND owtp.tour_id = perf.tour_id
    AND owtp.supplier_id = perf.supplier_id
  
  -- JOIN: Similar title detection
  LEFT JOIN (
    SELECT 
      owtp1.tour_id,
      owtp1.tour_option_id,
      owtp1.pricing_id,
      1 as has_similar_title,
      owtp2.tour_option_title as other_title,
      owtp2.from_price_eur as other_price_eur,
      ABS(owtp1.from_price_eur - owtp2.from_price_eur) / LEAST(owtp1.from_price_eur, owtp2.from_price_eur) as price_diff_pct
    FROM options_with_tour_percentiles owtp1
    INNER JOIN options_with_tour_percentiles owtp2
      ON owtp1.tour_id = owtp2.tour_id
      AND owtp1.tour_option_id != owtp2.tour_option_id
      AND owtp1.pricing_id != owtp2.pricing_id
      AND SUBSTRING(owtp1.tour_option_title, 1, 50) = SUBSTRING(owtp2.tour_option_title, 1, 50)
    WHERE ABS(owtp1.from_price_eur - owtp2.from_price_eur) / LEAST(owtp1.from_price_eur, owtp2.from_price_eur) > 0.3
  ) similar_title
    ON owtp.tour_id = similar_title.tour_id
    AND owtp.tour_option_id = similar_title.tour_option_id
    AND owtp.pricing_id = similar_title.pricing_id
  
  WHERE (
    -- Trigger A: Duplicate configs
    (title_stats.num_pricing_configs > 1 AND title_stats.num_distinct_prices > 1 
     AND title_stats.max_price_eur > title_stats.min_price_eur * 1.2)
    
    -- Trigger B: High price, no/low demand
    OR (COALESCE(perf.bookings_l12mo, 0) = 0 AND owtp.from_price_eur > owtp.tour_avg_price_eur * 1.2)
    OR (COALESCE(perf.bookings_l12mo, 0) > 0 AND COALESCE(perf.bookings_l12mo, 0) <= 5 
        AND owtp.from_price_eur > owtp.tour_avg_price_eur * 1.5)
    
    -- Trigger C: Similar titles, different prices
    OR (similar_title.has_similar_title = 1 AND similar_title.price_diff_pct > 0.3)
    
    -- Trigger D: High percentile + variance (original)
    OR (owtp.num_options > 1 AND owtp.tour_price_spread_ratio > 0.5 AND owtp.price_percentile_within_tour >= 0.75)
  )
),

-- BUCKET 3: FROM PRICE VS RETAIL PRICE MISMATCH
bucket3_price_mismatch AS (
  SELECT 
    supplier_id,
    segment,
    tour_id,
    tour_option_id,
    pricing_id,
    tour_title,
    tour_option_title,
    location_name,
    activity_type,
    tour_category,
    currency_iso_code,
    from_price_without_deal,
    from_price_eur,
    exchange_rate,
    has_individual_pricing,
    has_group_pricing,
    
    'BUCKET_3_PRICE_MISMATCH' as issue_bucket,
    NULL as sub_bucket,
    
    CASE 
      WHEN from_price_eur > base_retail_price_eur * 1.05 
      THEN CONCAT('From price (', ROUND(from_price_without_deal, 0), ' ', currency_iso_code, 
                  ') is ', ROUND(100.0 * (from_price_eur - base_retail_price_eur) / base_retail_price_eur, 1),
                  '% HIGHER than ', pricing_type_used, ' retail price (', ROUND(base_retail_price, 0), ' ', currency_iso_code, ')')
      WHEN from_price_eur < base_retail_price_eur * 0.95 
      THEN CONCAT('From price (', ROUND(from_price_without_deal, 0), ' ', currency_iso_code, 
                  ') is ', ROUND(100.0 * (base_retail_price_eur - from_price_eur) / base_retail_price_eur, 1),
                  '% LOWER than ', pricing_type_used, ' retail price (', ROUND(base_retail_price, 0), ' ', currency_iso_code, ')')
      ELSE NULL
    END as problem_description,
    
    CASE 
      WHEN from_price_eur > base_retail_price_eur * 1.05 THEN 10
      WHEN from_price_eur < base_retail_price_eur * 0.95 THEN 8
      ELSE 0
    END as severity_score,
    
    NULL as benchmark_median_eur,
    NULL as benchmark_p95_eur,
    pricing_type_used,
    NULL as tour_min_price_eur,
    NULL as tour_max_price_eur,
    NULL as tour_price_spread_pct,
    
    base_retail_price,
    base_retail_price_eur,
    ROUND(100.0 * (from_price_eur - base_retail_price_eur) / base_retail_price_eur, 1) as price_gap_pct
    
  FROM options_with_tour_percentiles
  WHERE base_retail_price_eur IS NOT NULL
    AND (from_price_eur > base_retail_price_eur * 1.05 
         OR from_price_eur < base_retail_price_eur * 0.95)
),

-- COMBINE ALL BUCKETS
all_issues AS (
  SELECT * FROM bucket1_high_from_prices
  UNION ALL
  SELECT * FROM bucket2_tour_issues
  UNION ALL
  SELECT * FROM bucket3_price_mismatch
)

-- FINAL OUTPUT
SELECT 
  issue_bucket,
  sub_bucket,
  severity_score,
  supplier_id,
  segment,
  tour_id,
  tour_option_id,
  pricing_id,
  tour_title,
  tour_option_title,
  location_name,
  activity_type,
  tour_category,
  has_individual_pricing,
  has_group_pricing,
  pricing_type_used,
  problem_description,
  
  currency_iso_code,
  from_price_without_deal as current_from_price,
  ROUND(from_price_eur, 2) as from_price_eur,
  ROUND(exchange_rate, 4) as exchange_rate_to_eur,
  
  -- Bucket-specific fields
  ROUND(benchmark_median_eur, 2) as benchmark_median_eur,
  ROUND(benchmark_p95_eur, 2) as benchmark_p95_eur,
  ROUND(tour_min_price_eur, 2) as tour_min_from_price_eur,
  ROUND(tour_max_price_eur, 2) as tour_max_from_price_eur,
  tour_price_spread_pct,
  base_retail_price,
  ROUND(base_retail_price_eur, 2) as base_retail_price_eur,
  price_gap_pct
  
FROM all_issues
WHERE problem_description IS NOT NULL
ORDER BY issue_bucket, sub_bucket, severity_score DESC, tour_id, tour_option_id;
